# CS 5330 — Week 5 In-Class Activity: Segment Anything on Your Produce Photos

Welcome back! In the pre-class reading you followed Chapter 50 from class segmentation to instance segmentation, and you saw that both still rely on a fixed list of class names. Today you'll work with the **Segment Anything** family of models from Meta, which loosens that constraint in two different ways:

| Model | How you prompt it | What you get back |
|---|---|---|
| **SAM** (2023) | A click or a box | One mask for whatever you pointed at, with **no class label** |
| **SAM 3** (2025) | A short noun phrase, such as `"apple"` | A separate mask for **every instance** of that concept |

You'll run both on the photos our class collected in Week 2. Then you'll revisit a question from Week 3: our CNN often confused **plums and apples**. Does SAM 3 have the same problem?

**Plan for today (about 75 minutes)**

1. Skim the reading (10 min)
2. Setup and data (10 min, mostly waiting on downloads, so start these cells right away)
3. Part A: SAM with click prompts (15 min)
4. Part B: SAM 3 with text prompts (15 min)
5. Part C: The plum vs. apple check (15 min)
6. Part D: Your highlights (10 min)

**Before you start:** select **Runtime → Change runtime type → T4 GPU**.

## Reading (10 minutes)

Read Meta's announcement of SAM 3: [Introducing Meta Segment Anything Model 3](https://ai.meta.com/blog/segment-anything-model-3/). You don't need every detail. Focus on three things:

- What "promptable concept segmentation" means, and how it differs from clicking on one object
- What kinds of prompts SAM 3 accepts
- How Meta built the training data

> **Tip:** Run the setup cells below *first*, then read while the models download.

## 1. Setup

SAM 3 support requires a recent version of Hugging Face `transformers`, so the first cell upgrades it. If Colab asks you to restart the session afterwards, restart and continue from the next cell.

In [ ]:
!pip install -q -U transformers accelerate

In [ ]:
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("⚠️ No GPU detected. Select Runtime → Change runtime type → T4 GPU, then rerun this cell.")

### Hugging Face access for SAM 3

The SAM 3 weights are **gated**: Meta asks you to accept their licence before downloading. You should have done this before class (see the Week 5 pre-class page). If you haven't, do it now:

1. Sign in at [huggingface.co](https://huggingface.co) and open the [facebook/sam3](https://huggingface.co/facebook/sam3) model page.
2. Accept the licence terms on that page. Approval is not always instant.
3. Create a **read** token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
4. In Colab, click the 🔑 **Secrets** icon in the left sidebar, add a secret named `HF_TOKEN` with your token as the value, and enable notebook access.

Please never paste a token directly into a code cell. Notebooks get shared, and the token goes with them.

Part A uses the original SAM, which is not gated, so you can start there while you wait for approval.

In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("Logged in to Hugging Face using the HF_TOKEN secret.")
except Exception as err:  # Secret missing, notebook access disabled, or not running in Colab
    print(f"Could not read the HF_TOKEN secret ({type(err).__name__}).")
    print("Part A will still work; Parts B and C need SAM 3 access.")

## 2. Get the class dataset

The full class repo is large, so instead of cloning all of it, we do a **sparse checkout**: Git downloads the folder list first, then only the image files for the classes we ask for. Today that's apple and plum.

In [ ]:
REPO_URL = "https://github.com/LinoNU/cs5330-produce-photos.git"
REPO_DIR = Path("produce_photos")
DATASET_FOLDER = "CS 5330 Fall 2026 Class Dataset"   # Images live in this subfolder of the repo
CLASSES = ["apple", "plum"]


def run(cmd):
    """Run a shell command (given as a list, so folder names with spaces are safe)."""
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}\n{result.stderr}")
    return result.stdout


if not REPO_DIR.exists():
    run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", REPO_URL, str(REPO_DIR)])

# List the class folders without downloading any images yet
listing = run(["git", "-C", str(REPO_DIR), "ls-tree", "-d", "--name-only", "HEAD", f"{DATASET_FOLDER}/"])
class_folders = [Path(line).name for line in listing.splitlines()]
print("Class folders in the repo:", class_folders)

# Match folders by prefix, case-insensitively (e.g., "Apple" or "apples" both match "apple")
selected = {}
for cls in CLASSES:
    matches = [f for f in class_folders if f.lower().startswith(cls)]
    if not matches:
        raise ValueError(f"No folder found for '{cls}'. Check the folder list printed above.")
    selected[cls] = matches[0]

run(["git", "-C", str(REPO_DIR), "sparse-checkout", "set",
     *[f"{DATASET_FOLDER}/{folder}" for folder in selected.values()]])
print("Downloaded:", selected)

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

image_paths = {
    cls: sorted(p for p in (REPO_DIR / DATASET_FOLDER / folder).iterdir()
                if p.suffix.lower() in IMAGE_EXTENSIONS)
    for cls, folder in selected.items()
}
for cls, paths in image_paths.items():
    print(f"{cls}: {len(paths)} images")


def load_image(path, max_side=1024):
    """Load an RGB image and shrink it so its longer side is at most max_side pixels.

    Phone photos can be 12+ megapixels. Shrinking them first keeps the masks
    (which are returned at full image size) from filling up GPU memory.
    """
    image = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    image.thumbnail((max_side, max_side))
    return image


# Quick look at a few examples from each class
fig, axes = plt.subplots(len(CLASSES), 4, figsize=(12, 3 * len(CLASSES)))
for row, cls in enumerate(CLASSES):
    for col, path in enumerate(image_paths[cls][:4]):
        axes[row, col].imshow(load_image(path))
        axes[row, col].set_title(f"{cls}: {path.name}", fontsize=8)
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def show_masks(image, masks, scores=None, title="", ax=None):
    """Overlay each binary mask on the image in its own colour."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(image)
    colours = plt.cm.tab10.colors
    for i, mask in enumerate(masks):
        mask = np.asarray(mask, dtype=bool)
        overlay = np.zeros((*mask.shape, 4))
        overlay[mask] = (*colours[i % len(colours)], 0.5)
        ax.imshow(overlay)
    if scores is not None and len(scores) > 0:
        title += "\nscores: " + ", ".join(f"{float(s):.2f}" for s in scores)
    ax.set_title(title, fontsize=9)
    ax.axis("off")
    return ax

## Part A: SAM with a click (15 minutes)

The original SAM takes a **point** (or a box) and returns a mask for the object under it. A single click is ambiguous, though. Are you pointing at the whole apple, its skin, or just the stem? SAM handles this by returning **three candidate masks**, each with a predicted quality score.

We'll use the smallest variant, `sam-vit-base`, which is fast on a T4.

In [ ]:
from transformers import SamModel, SamProcessor

sam1_model = SamModel.from_pretrained("facebook/sam-vit-base").to(device).eval()
sam1_processor = SamProcessor.from_pretrained("facebook/sam-vit-base")


def sam1_click(image, x, y):
    """Segment the object at pixel (x, y). Returns 3 candidate masks and their predicted IoU scores."""
    inputs = sam1_processor(image, input_points=[[[x, y]]], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = sam1_model(**inputs)
    masks = sam1_processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(), inputs["original_sizes"].cpu(), inputs["reshaped_input_sizes"].cpu()
    )[0][0]                              # shape: (3, H, W)
    scores = outputs.iou_scores.cpu()[0, 0]  # shape: (3,)
    return masks, scores

In [ ]:
# 🎛️ Try different images and click locations. (0, 0) is the top-left corner.
image = load_image(image_paths["apple"][0])
x, y = image.width // 2, image.height // 2   # Start with the centre of the image

masks, scores = sam1_click(image, x, y)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, ax in enumerate(axes):
    show_masks(image, [masks[i]], [scores[i]], title=f"Candidate {i + 1}", ax=ax)
    ax.plot(x, y, marker="*", color="red", markersize=15)
plt.tight_layout()
plt.show()

**Try this:**

- Move the point onto the stem, a leaf, a highlight, or the background. How do the three candidates change?
- Look at what SAM returned. Did it ever tell you *what* the object is?

### Segment everything

SAM can also run without any prompt: it places a grid of points over the image and keeps every distinct mask it finds. This is slower, so try it on just one or two images.

In [ ]:
from transformers import pipeline

mask_generator = pipeline("mask-generation", model="facebook/sam-vit-base",
                          device=0 if device == "cuda" else -1)

image = load_image(image_paths["plum"][0])
result = mask_generator(image, points_per_batch=64)
print(f"SAM found {len(result['masks'])} masks")
show_masks(image, result["masks"], title="Segment everything")
plt.show()

**Think about it:** Of all the masks SAM found, how many are actually fruit? For an inspection line, is "segment everything" useful on its own, or do you still need something else?

## Part B: SAM 3 with text prompts (15 minutes)

SAM 3 takes a **concept**, a short noun phrase, and returns a mask for every instance of it, each with its own confidence score. This combines the instance segmentation from Section 50.6 with an open vocabulary: there's no fixed list of classes.

SAM 3 is much larger than `sam-vit-base` (about 850M parameters), so loading takes a minute or two.

In [ ]:
from transformers import Sam3Model, Sam3Processor

sam3_model = Sam3Model.from_pretrained("facebook/sam3").to(device).eval()
sam3_processor = Sam3Processor.from_pretrained("facebook/sam3")

SCORE_THRESHOLD = 0.5   # 🎛️ Detections below this confidence are discarded
MASK_THRESHOLD = 0.5


def sam3_text(image, prompts):
    """Run one or more text prompts on an image.

    The image is encoded once and reused for every prompt, which is much faster
    than re-encoding it per prompt. Returns {prompt: {"masks", "boxes", "scores"}}.
    """
    img_inputs = sam3_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        vision_embeds = sam3_model.get_vision_features(pixel_values=img_inputs.pixel_values)

    results = {}
    for prompt in prompts:
        text_inputs = sam3_processor(text=prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = sam3_model(vision_embeds=vision_embeds, **text_inputs)
        results[prompt] = sam3_processor.post_process_instance_segmentation(
            outputs,
            threshold=SCORE_THRESHOLD,
            mask_threshold=MASK_THRESHOLD,
            target_sizes=img_inputs.get("original_sizes").tolist(),
        )[0]
    return results

In [ ]:
# 🎛️ Change the image and the prompts. Keep prompts short: one noun or a noun phrase.
image = load_image(image_paths["apple"][0])
prompts = ["apple", "stem", "leaf", "bruise"]

results = sam3_text(image, prompts)

fig, axes = plt.subplots(1, len(prompts), figsize=(5 * len(prompts), 5))
for ax, prompt in zip(axes, prompts):
    r = results[prompt]
    show_masks(image, r["masks"].cpu(), r["scores"].cpu(),
               title=f'"{prompt}": {len(r["scores"])} found', ax=ax)
plt.tight_layout()
plt.show()

**Try this:**

- Find a photo with more than one fruit in it. Does SAM 3 give each one its own mask, even when they touch? (This is your reflection's question 2, live.)
- Try a prompt for something that *isn't* in the image. Does SAM 3 correctly return nothing?
- Try more specific phrases, like `"red apple"` or `"apple with a sticker"`. Where does it break down?

## Part C: The plum vs. apple check (15 minutes)

In Week 3, our transfer-learning CNN often confused plums and apples. Let's ask SAM 3 the same question. For every apple and plum photo, we prompt with both `"apple"` and `"plum"`, and record the highest confidence score for each prompt.

One important difference: the CNN was a **forced-choice** classifier, so its softmax had to pick a class. SAM 3 is not forced to choose. It can find an apple, a plum, both, or neither. Keep this in mind when comparing the two.

In [ ]:
MAX_PER_CLASS = 30   # 🎛️ Lower this if you're short on time
PROMPTS = ["apple", "plum"]

records = []
for true_label in CLASSES:
    for path in image_paths[true_label][:MAX_PER_CLASS]:
        results = sam3_text(load_image(path), PROMPTS)
        row = {"file": path.name, "true": true_label}
        for prompt in PROMPTS:
            scores = results[prompt]["scores"]
            row[f"max_score_{prompt}"] = float(scores.max()) if len(scores) > 0 else 0.0
        records.append(row)

df = pd.DataFrame(records)


def outcome(row):
    found_apple = row["max_score_apple"] > 0
    found_plum = row["max_score_plum"] > 0
    if found_apple and found_plum:
        return "both"
    if found_apple:
        return "apple only"
    if found_plum:
        return "plum only"
    return "neither"


df["outcome"] = df.apply(outcome, axis=1)
pd.crosstab(df["true"], df["outcome"], margins=True)

In [ ]:
# Forced-choice view, to compare with the Week 3 CNN: pick whichever prompt scored higher
df["forced_choice"] = np.where(df["max_score_apple"] >= df["max_score_plum"], "apple", "plum")
df.loc[(df["max_score_apple"] == 0) & (df["max_score_plum"] == 0), "forced_choice"] = "neither"

accuracy = (df["forced_choice"] == df["true"]).mean()
print(f"Forced-choice accuracy: {accuracy:.0%}\n")
pd.crosstab(df["true"], df["forced_choice"])

In [ ]:
# Look at the hardest cases: plums with the highest "apple" score, and vice versa
for true_label, wrong_prompt in [("plum", "apple"), ("apple", "plum")]:
    worst = (df[df["true"] == true_label]
             .sort_values(f"max_score_{wrong_prompt}", ascending=False)
             .head(4))
    fig, axes = plt.subplots(1, len(worst), figsize=(4 * len(worst), 4), squeeze=False)
    for ax, (_, row) in zip(axes[0], worst.iterrows()):
        image = load_image(REPO_DIR / DATASET_FOLDER / selected[true_label] / row["file"])
        r = sam3_text(image, [wrong_prompt])[wrong_prompt]
        show_masks(image, r["masks"].cpu(), r["scores"].cpu(),
                   title=f'true: {true_label} | "{wrong_prompt}"', ax=ax)
    plt.suptitle(f"{true_label.title()} photos that SAM 3 most readily calls '{wrong_prompt}'")
    plt.tight_layout()
    plt.show()

**Think about it:**

- Compare this with your Week 3 results. Is SAM 3 better, worse, or confused in a different way?
- How many images landed in "both"? What does that tell you about how SAM 3 decides, compared with a softmax classifier?
- Change `SCORE_THRESHOLD` and rerun Part C. How sensitive are the results to that one number? (Think back to the precision/recall tradeoff from Week 4.)

## Part D: Your highlights (10 minutes)

Write down **four highlights** from today, one of each kind below. Be specific: name the image, the prompt, or the result you're referring to.

1. **A capability that impressed you.** Something SAM or SAM 3 did better than you expected.
2. **A failure case.** A prompt or image where it went wrong. What do you think caused it?
3. **A connection to the reading.** Link something you saw today to Chapter 50 (for example, one of the "shortcomings" sections) or to the SAM 3 announcement.
4. **An implication for the inspection line.** Would you deploy SAM 3 for produce inspection? For which task, and what would you need to check first?

Write your answers in the cell below, then submit them on Canvas before the end of class.

*Your highlights:*

1.
2.
3.
4.